# Pronunciation Coach: fine-tune the phoneme recognizer (v2-3)

Fine-tunes `facebook/wav2vec2-lv-60-espeak-cv-ft` so that it outputs the sounds a speaker **actually said**,
including pronunciation errors, instead of the sounds they should have said.

- **Data:** the private Kaggle dataset made from `pronunciation-coach-finetune.zip` (see `scripts/package_kaggle.py`):
  L2-ARCTIC train/dev speakers (what the annotators heard) and CMU ARCTIC native US speakers.
- **Settings:** Accelerator **GPU T4 x1** (one GPU is enough), **Internet on** (to download the base model).
- **Time:** at most about 3 hours; training stops early when the dev F1 stops improving.
- **Output:** `/kaggle/working/recognizer-l2arctic.zip` (the best checkpoint + training log).

License: L2-ARCTIC is CC BY-NC 4.0, so the fine-tuned model is for non-commercial use only.
The base model is Apache-2.0. No token is needed; if one ever is, add it under *Add-ons → Secrets* as `HF_TOKEN`.

In [ ]:
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - turn on the GPU accelerator!")

In [ ]:
!pip -q install soundfile

In [ ]:
from pathlib import Path

# Kaggle unpacks the uploaded zip; find the folder that holds manifests/
DATA = next(p.parent.parent for p in Path("/kaggle/input").rglob("manifests/l2arctic_train.jsonl"))
OUT = Path("/kaggle/working/recognizer")
print(DATA)
print(open(DATA / "README.txt").read())

## Training
One epoch = twice the non-native sentences, sampled so that non-native and native speech weigh the same.
After every epoch the script prints the dev metrics (PER, MDD F1, native PER) and keeps the best checkpoint by F1.

In [ ]:
!PYTHONDONTWRITEBYTECODE=1 python {DATA}/scripts/finetune_recognizer.py --data {DATA} --out {OUT}

## Results

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

log = pd.DataFrame([json.loads(line) for line in open(OUT / "train_log.jsonl")])
display(log)
loss = pd.read_csv(OUT / "loss.csv")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(loss.step, loss.loss.rolling(20, min_periods=1).mean()); ax[0].set_title("training loss (CTC)"); ax[0].set_xlabel("step")
ax[1].plot(log.epoch, log.l2_dev_f1, marker="o", label="L2-ARCTIC dev MDD F1")
ax[1].plot(log.epoch, log.l2_dev_per, marker="o", label="L2-ARCTIC dev PER")
ax[1].plot(log.epoch, log.native_dev_per, marker="o", label="native dev PER"); ax[1].legend(); ax[1].set_xlabel("epoch")
plt.savefig(OUT / "curves.png", dpi=100, bbox_inches="tight"); plt.show()

In [ ]:
import shutil

best = OUT / "best"
if best.exists():
    for name in ("train_log.jsonl", "loss.csv", "curves.png"):
        shutil.copy(OUT / name, best / name)
    archive = shutil.make_archive("/kaggle/working/recognizer-l2arctic", "zip", best)
    print(archive, round(Path(archive).stat().st_size / 1e6), "MB")
else:
    print("No epoch beat the original model: nothing to download (see the log above).")